# Breeze ASR 25 — exp5 領域外對照

**為什麼要跑這個:** Breeze ASR 25 的訓練資料含 `NTUML2021`(11 小時),
那正是 exp2 的 S1/S2 來源課程——**模型在我們的評測語料上微調過**。
所以 Phase A 那些漂亮數字不能拿來說「Breeze 比較好」。

exp5 換一批模型沒看過的語料:台灣科技訪談 podcast 的三段(開源社群/搜尋引擎後端、
數位身分/網路服務、資安產業),語域完全離開 ML,術語密度與 exp2 同級。

**你要做的事:執行階段 → 全部執行。就這樣。**
這一版**不需要雲端硬碟**——音檔來自公開 RSS,notebook 自己抓。

## 1. 環境與 GPU 檢查

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "(無輸出)")
import torch
if not torch.cuda.is_available():
    raise RuntimeError("沒有 GPU。執行階段 → 變更執行階段類型 → T4 GPU,再重跑。")
print("GPU OK:", torch.cuda.get_device_name(0))

%pip install -q "transformers>=4.44" librosa soundfile imageio-ffmpeg
import transformers, librosa
print("transformers", transformers.__version__, "| torch", torch.__version__)

## 2. 參數

In [ ]:
REPO     = "clarencechien/kikemu"
BRANCH   = "claude/improve-experiment-credibility-c8ekb2"
MODEL_ID = "MediaTek-Research/Breeze-ASR-25"
REVISION = "cffe7ccb404d025296a00758d0a33468bec3a9d0"
CONDS    = ["M0", "M3"]   # 乾淨 / 混響+交談 12dB(handoff-v6 §3:只取兩端)
# language 參數在 Phase A 實測對輸出**位元無影響**(模型 forced_decoder_ids
# 的語言槽是 None,自動偵測本來就判中文),所以這裡只跑一次,不重複燒 GPU 時間。
# 下面仍保留一次 language="en" 的健全性檢查,確認參數真的有被吃進去。

## 3. 取語料 → 加噪 → 對指紋

全部由 repo 裡的腳本完成,notebook 不自己實作:

- `exp5/scripts/prep_audio.py` — 依 `picks.json` 從 RSS 抓 MP3、切出選定視窗
- `exp5/scripts/degrade.py` — 呼叫 **exp2 的** `degrade.py::build()` 產生 M0–M3
  (同一顆 RIR、同一份 DEMAND 噪音、同一組種子,與 exp2 才是同一把尺)

最後對 `exp5/corpus/audio_manifest.json` 的 SHA256——指紋相符就代表你手上的
音檔與 SM / Gemini 兩個 arm 跑的是同一份。

In [ ]:
import hashlib, json, os, subprocess, sys
from pathlib import Path

WORK = Path("/content/kikemu")
if WORK.exists():
    # 已經 clone 過就拉最新的——不然修正推上去了,你這邊還是舊版。
    subprocess.run(["git", "-C", str(WORK), "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", str(WORK), "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    f"https://github.com/{REPO}.git", str(WORK)], check=True)
os.chdir(WORK)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

for script in ("exp5/scripts/prep_audio.py", "exp5/scripts/degrade.py"):
    r = subprocess.run([sys.executable, script], cwd=WORK, capture_output=True, text=True)
    print(r.stdout.strip())
    if r.returncode != 0:
        print(r.stderr.strip())
        raise RuntimeError(f"{script} 失敗,原因見上方 stderr")

MAN = json.loads((WORK / "exp5/corpus/audio_manifest.json").read_text())
picks = json.loads((WORK / "exp5/corpus/picks.json").read_text())
targets = [f"{p['seg']}__{c}" for p in picks for c in CONDS]
print("\n聲學條件指紋:")
for stem in targets:
    p = WORK / "exp5/corpus/conditions" / f"{stem}.wav"
    ok = sha256(p) == MAN["conditions"][stem]["sha256"]
    print(f"  {stem}  {'✅ bit-identical' if ok else '⚠️ 不符'}")
print(f"\n本次 {len(targets)} 個檔")


## 4. 載入模型(`chunk_length_s=0`,模型卡建議的 sequential)

In [ ]:
import time, torch
from transformers import (AutomaticSpeechRecognitionPipeline, WhisperForConditionalGeneration,
                          WhisperProcessor)
t0 = time.time()
processor = WhisperProcessor.from_pretrained(MODEL_ID, revision=REVISION)
model = WhisperForConditionalGeneration.from_pretrained(
    MODEL_ID, revision=REVISION, torch_dtype=torch.float16).to("cuda").eval()
asr = AutomaticSpeechRecognitionPipeline(
    model=model, tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    chunk_length_s=0, torch_dtype=torch.float16, device="cuda")
assert asr._preprocess_params.get("chunk_length_s") in (0, None), "chunk_length_s 沒生效"
print(f"載入完成 {time.time()-t0:.0f}s | revision={REVISION[:12]}")

## 5. 推論

輸出欄位對齊既有 arm(`score.py` 讀 `transcript`,檔名 `<seg>__<cond>.json`),
每筆並記下音檔 sha256,結果 JSON 自己就能證明跑的是哪一份音檔。

In [ ]:
import json, time, librosa
RAW = WORK / "exp5/results/raw/Xbrz_auto"
RAW.mkdir(parents=True, exist_ok=True)
meta_env = {"model": MODEL_ID, "revision": REVISION, "dtype": "float16",
            "chunk_length_s": 0, "return_timestamps": True, "language_arg": None,
            "transformers": transformers.__version__, "torch": torch.__version__,
            "gpu": torch.cuda.get_device_name(0), "corpus": "exp5"}
(RAW / "_meta.json").write_text(json.dumps(meta_env, ensure_ascii=False, indent=1))

for stem in targets:
    dst = RAW / f"{stem}.json"
    if dst.exists():
        print(f"  跳過(已有){stem}"); continue
    wav = WORK / "exp5/corpus/conditions" / f"{stem}.wav"
    audio, _ = librosa.load(wav, sr=16000, mono=True)
    t0 = time.time()
    r = asr(audio.copy(), return_timestamps=True)
    el = time.time() - t0
    dst.write_text(json.dumps({
        "arm": "Xbrz_auto", "file": f"{stem}.wav",
        "audio_s": round(len(audio) / 16000, 1),
        "transcript": r["text"].strip(),
        "segments": [{"start": c["timestamp"][0], "end": c["timestamp"][1], "text": c["text"]}
                     for c in r.get("chunks", [])],
        "meta": {**meta_env, "elapsed_sec": round(el, 1), "audio_sha256": sha256(wav),
                 "audio_matches_manifest": sha256(wav) == MAN["conditions"][stem]["sha256"]},
    }, ensure_ascii=False, indent=1))
    print(f"  {stem}  {el:.0f}s  {len(r['text'])} 字")

# language 參數健全性:Phase A 量到 auto 與 zh 位元相同,要排除「參數根本沒生效」
stem = targets[0]
audio, _ = librosa.load(WORK / "exp5/corpus/conditions" / f"{stem}.wav", sr=16000, mono=True)
base = json.loads((RAW / f"{stem}.json").read_text())["transcript"]
en = asr(audio.copy(), return_timestamps=True,
         generate_kwargs={"language": "en", "task": "transcribe"})["text"].strip()
(RAW / "_lang_sanity.json").write_text(json.dumps(
    {"stem": stem, "en_differs": en != base, "en_head": en[:300]}, ensure_ascii=False, indent=1))
print("\nlanguage 健全性:改成 en 後輸出",
      "有變化 → 參數確實生效 ✅" if en != base else "仍相同 → 參數可能沒被吃進去 ⚠️")

## 6. 看一眼 + 帶走

In [ ]:
from IPython.display import Markdown, display
terms = json.loads((WORK / "exp5/corpus/terms.json").read_text())
lines = []
for stem in targets:
    seg = stem.split("__")[0]
    d = json.loads((RAW / f"{stem}.json").read_text())
    want = [t["term"] for t in terms[seg]]
    hit = [t for t in want if t.lower() in d["transcript"].lower()]
    lines.append(f"### {stem} — 字面命中 {len(hit)}/{len(want)}({d['meta']['elapsed_sec']}s)")
    lines.append("> " + d["transcript"][:400] + ("…" if len(d["transcript"]) > 400 else ""))
display(Markdown("\n\n".join(lines)))

import shutil
z = shutil.make_archive("/content/xbrz-exp5", "zip", WORK / "exp5/results/raw", ".")
print("已打包:", z)
try:
    from google.colab import files; files.download(z)
except Exception as e:
    print("自動下載失敗,請從左側檔案面板下載:", e)